# Predicting Hotel Data - ISA 444 Final Project

## Project Purpose

The goal of this project is to predict hotel occupancy. We are testing multiple models on a holdout of the data to see which one performs the best based on MAE.

## Data Description

The sample of hotel data is for 18 different hotels. The data set includes the unique id for each hotel, date stamp, day, month, year, and percent occupied as a percentage. There are some hotels that when full occupancy is reached, convert conference areas into additional hotel rooms to house more guests, which causes the percent occupancy to surpass 1.0.

Installing Packages

In [ ]:
!pip install utilsforecast

In [ ]:
!pip install statsforecast

In [ ]:
!pip install lightgbm

In [ ]:
!pip install NeuralForecast

In [ ]:
!pip install nixtla

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.6/127.6 kB 8.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from utilsforecast.plotting import plot_series
from statsforecast import StatsForecast
from statsforecast.models import HistoricAverage, Naive, SeasonalNaive, WindowAverage, SeasonalWindowAverage, AutoARIMA,AutoETS
from utilsforecast.losses import rmse, mae, mape
from utilsforecast.evaluation import evaluate
import lightgbm as lgb
from neuralforecast.auto import AutoNBEATS, AutoNHITS
from neuralforecast import NeuralForecast

Reading in the Data

In [ ]:
df = pd.read_parquet("/sample_hotels.parquet")

df = (
    df
    .assign(
        ds = lambda x: pd.to_datetime(x['ds']),
        day = lambda x: x['day'].astype('category'),
        month = lambda x: x['month'].astype('category'),
        year = lambda x: x['year'].astype('category')
    )
      #making dataset exactly 1 year
)

validation = df.query('ds > "2023-01-01" ')
df = df.query('ds <= "2023-01-01" ')

In [ ]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6588 entries, 1430 to 297861
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   unique_id  6588 non-null   object        
 1   ds         6588 non-null   datetime64[us]
 2   day        6588 non-null   category      
 3   month      6588 non-null   category      
 4   year       6588 non-null   category      
 5   y          6588 non-null   float64       
dtypes: category(3), datetime64[us](1), float64(1), object(1)
memory usage: 226.0+ KB


In [ ]:
print(df['unique_id'].nunique())
unique_ids = df['unique_id'].unique()
print(unique_ids)

18
['hotel_0' 'hotel_7' 'hotel_14' 'hotel_21' 'hotel_28' 'hotel_35'
 'hotel_42' 'hotel_49' 'hotel_56' 'hotel_63' 'hotel_70' 'hotel_84'
 'hotel_91' 'hotel_98' 'hotel_105' 'hotel_112' 'hotel_126' 'hotel_133']


## Simple Models

In [ ]:
simple_models = [
    Naive(),
    SeasonalNaive(season_length=7, alias = 'weekly_seasonality'),
    SeasonalNaive(season_length=28, alias = 'monthly_seasonality'),
    AutoETS(season_length = 7, alias = 'AutoETS'),
    AutoARIMA(season_length = 7, alias = 'AutoARIMA')
]

simple_sf = StatsForecast(
    models = simple_models,
    freq='D'
)

simple_cv = simple_sf.cross_validation(
    h = 28,
    df = df[['unique_id', 'ds', 'y']],
    n_windows = 12, # step*window = 28*12 = whole year
    step_size = 28
)

simple_eval = evaluate(
    df = simple_cv,
    metrics = [mae, rmse, mape],
    models = simple_cv.columns[4:].tolist()
)

simple_eval

KeyboardInterrupt: 

## Light gbm

In [ ]:
!pip install mlforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 2.3 MB/s eta 0:00:00


In [ ]:
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean


df_ml = pd.get_dummies(df, columns = ['day', 'month', 'year'], drop_first=True)

ml_models = [
    lgb.LGBMRegressor()
]

mlf = MLForecast(
    models=ml_models,
    freq="D",
    lags=(7, 28),
    lag_transforms={
        7: [RollingMean(window_size=7)]
    }
)

cross_validation_mlf = mlf.cross_validation(
    df=df_ml,
    n_windows=6,
    h=28,
    id_col='unique_id',
    time_col='ds',
    target_col='y',
    static_features=[]
)

ml_eval = evaluate(
    df = cross_validation_mlf,
    metrics = [mae, rmse, mape],
    models = cross_validation_mlf.columns[4:].tolist()
)

display(ml_eval)


ModuleNotFoundError: No module named 'mlforecast'

## Neural Net Models - NBEATS & NHITS

In [ ]:
#Took 43 minutes to run with 1 CPU, don't run if you don't have to, otherwise change num_samples to something smaller
from neuralforecast.losses.pytorch import MQLoss

def small_config(trial):
    return {
        "input_size": trial.suggest_categorical("input_size", [14, 28]),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2),
        "scaler_type": trial.suggest_categorical("scaler_type", ["standard"])
    }

nn_models = [
    AutoNBEATS(
        h=28,
        loss=MQLoss(level=[80, 20]),
        config=small_config,
        num_samples=5,
        backend='optuna',
        alias='AutoNBEATS'
    ),
    AutoNHITS(
        h=28,
        loss=MQLoss(level=[80, 20]),
        config=small_config,
        num_samples=5,
        backend='optuna',
        alias='AutoNHITS'
    ),
]

nn_forecast = NeuralForecast(
    models=nn_models,
    freq='D'
)
#Cross validation

cross_validation_nn = nn_forecast.cross_validation(
    df = df,
    n_windows= 4,
    h = 28,
    step_size = 28
)

nn_eval = evaluate(
    df=cross_validation_nn.drop(columns='cutoff'),
    metrics=[mae, rmse, mape]
)

[I 2025-11-24 20:03:10,610] A new study created in memory with name: no-name-1893f363-2269-4c9a-bc4d-501af6479fc0
INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MQLoss        | 5      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.6 M  | train
-------------------------------------------------------
2.6 M     Trainable params
2.4 K     Non-trainable params
2.6 M     Total params
10.575    Total estimated model params size (MB)
31        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=1000` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


Predicting: |          | 0/? [00:00<?, ?it/s]

## TimeGPT

In [ ]:
!pip install nixtla

In [ ]:
from nixtla import NixtlaClient
from nixtla.nixtla_client import TimeGPT
import os

transformer_models = [
    TimeGPT()
]
#Forecast


#Cross Validation

#Evaluation


ImportError: cannot import name 'TimeGPT' from 'nixtla.nixtla_client' (/usr/local/lib/python3.12/dist-packages/nixtla/nixtla_client.py)

## Evaluating

In [ ]:
#Single dataframe
combined_df = pd.concat(
    [
        simple_eval,
        ml_eval.drop(columns=['unique_id', 'metric']),
        nn_eval.drop(columns=['unique_id', 'metric'])
    ],
    axis=1
)

NameError: name 'simple_eval' is not defined

In [ ]:
evaluation = (
    combined_df
    .sort_values(by = 'unique_id')
    .reset_index(drop = True)
    .assign(best_model = lambda x: x.iloc[:, 2:].idxmin(axis=1))
)

evaluation_mae = (
    evaluation
    .query("metric == 'mae'")
    .reset_index(drop = True)
)

evaluation_mae.value_counts('best_model')

,count
best_model,
AutoNBEATS-median,6
AutoARIMA,4
LGBMRegressor,4
AutoETS,1
AutoNHITS-median,1
Naive,1
weekly_seasonality,1


In [ ]:
#Count how often each model wins based on MAE
def count_model_wins(forecast_df):
    model_columns = [col for col in forecast_df.columns if col not in ['ds', 'y', 'unique_id']]

    wins = {}
    for model in model_columns:
        wins[model] = 0

    for i in range(len(forecast_df)):
        actual = forecast_df['y'].iloc[i]
        errors = {}

        for model in model_columns:
            pred = forecast_df[model].iloc[i]
            errors[model] = abs(actual - pred)

        winner = min(errors, key=errors.get)
        wins[winner] += 1

    return wins

def calculate_mae_scores(forecast_df):
    from utilsforecast.losses import mae

    model_columns = [col for col in forecast_df.columns if col not in ['ds', 'y', 'unique_id']]

    mae_scores = {}
    for model in model_columns:
        mae_scores[model] = mae(forecast_df['y'], forecast_df[model])

    return mae_scores

In [ ]:
#Final eval from above -> save to CSV

def save_results_to_csv(forecast_df, filename='model_results.csv'):
    import pandas as pd

    mae_scores = calculate_mae_scores(forecast_df)

    wins = count_model_wins(forecast_df)

    results = []
    for model in mae_scores.keys():
        results.append({
            'Model': model,
            'MAE': mae_scores[model],
            'Wins': wins[model],
            'Win_Rate_%': (wins[model] / len(forecast_df)) * 100
        })

    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('MAE')
    results_df.to_csv(filename, index=False)
    print(f"Results saved to {filename}")

    return results_df

In [ ]:
#Plot forecasts and actuals

def plot_forecasts(forecast_df, title='Forecasts vs Actuals'):

    import matplotlib.pyplot as plt

    model_columns = [col for col in forecast_df.columns if col not in ['ds', 'y', 'unique_id']]

    plt.figure(figsize=(16, 8))

    plt.plot(forecast_df['ds'], forecast_df['y'],
             label='Actual', color='black', linewidth=3, marker='o',
             markersize=8, zorder=10)

    colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'pink', 'gray']
    for i, model in enumerate(model_columns):
        plt.plot(forecast_df['ds'], forecast_df[model],
                label=model, linestyle='--', marker='s',
                alpha=0.6, linewidth=2, markersize=5,
                color=colors[i % len(colors)])

    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Value', fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(loc='best', fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
